# Attention Complexity Optimizations — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import time
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

Setup

In [ ]:
import time
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Attention Complexity Optimizations — Lab

NumPy lab: KV cache + tiled attention (FlashAttention intuition).

You will compare:
- reference attention (materialize scores)
- KV-cached decode
- tiled attention (streaming softmax)

## Task 2: Section 0 — Stable softmax + reference attention

## Section 0 — Stable softmax + reference attention

Implement two building blocks:
- a numerically stable row-wise `softmax`
- a reference attention function that materializes the full score matrix

Use this section as the correctness baseline for the rest of the lab. The output should have shape `(T, d)`.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def attn_ref(Q: np.ndarray, K: np.ndarray, V: np.ndarray) -> np.ndarray:
    # Q: (Tq,d), K,V: (Tk,d)
    d = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d)
    A = softmax(scores, axis=1)
    return A @ V

T, d = 64, 32
Q = rng.standard_normal((T, d))
K = rng.standard_normal((T, d))
V = rng.standard_normal((T, d))
O = attn_ref(Q, K, V)
check('ref_shape', O.shape == (T, d))

In [ ]:
# Checks
check('ref_shape', O.shape == (T, d))

## Task 3: Section 1 — Prefill vs decode (work estimate)

## Section 1 — Prefill vs decode (work estimate)

Estimate how attention work scales in three cases:
- full prefill over a length-`T` sequence
- naive autoregressive decode that recomputes everything each step
- decode with a KV cache

The goal is not an exact FLOP count. The goal is to capture the right growth rates and compare them numerically.

In [ ]:
def flops_prefill(T: int, d: int) -> int:
    # Roughly: compute scores (T^2*d) + apply to V (T^2*d)
    return 2 * T * T * d

def flops_decode_naive(T: int, d: int) -> int:
    # If you recompute attention from scratch at each step t=1..T, cost ~ sum_{t=1..T} 2*t^2*d
    return int(2 * d * sum(t*t for t in range(1, T+1)))

def flops_decode_kvcache(T: int, d: int) -> int:
    # Each new token attends to all previous keys: per step O(t*d), total ~ sum 2*t*d
    return int(2 * d * sum(t for t in range(1, T+1)))

for T in [64, 256, 1024]:
    d = 64
    pf = flops_prefill(T, d)
    dn = flops_decode_naive(T, d)
    dk = flops_decode_kvcache(T, d)
    print('T', T, 'prefill', pf, 'decode_naive', dn, 'decode_kvcache', dk, 'speedup~', round(dn/max(1,dk), 1))

## Task 4: Section 2 — KV cache (incremental decoding)

## Section 2 — KV cache (incremental decoding)

Implement causal decoding one token at a time using cached keys and values.

Do two things:
- write a single-step decode function that attends over the cache
- write a sequential decode loop and verify it matches a causal reference implementation

If your cache logic is correct, the maximum absolute error should be essentially zero.

In [ ]:
$f

In [ ]:
# Checks
check('kv_matches_ref', err < 1e-8)

## Task 5: Section 3 — Tiled attention (FlashAttention intuition)

## Section 3 — Tiled attention (FlashAttention intuition)

Implement tiled attention using a streaming softmax over blocks of keys.

Important idea:
- never materialize the full `Tq x Tk` score matrix
- maintain a running max `m`, running normalizer `l`, and running weighted output `out`
- rescale old contributions whenever a new block changes the row max

Your tiled result should closely match the reference attention output.

In [ ]:
$11

In [ ]:
# Checks
check('tiled_close', err < 1e-6)

## Task 6: Section 4 — Simple timing (optional)

## Section 4 — Simple timing (optional)

Write a tiny benchmarking helper and compare the average runtime of:
- reference attention
- tiled attention

This timing is only illustrative because NumPy on CPU is not FlashAttention. The point is to build intuition for how implementation details affect runtime.

In [ ]:
def bench(fn, *args, n_runs: int = 10):
    t0 = time.time()
    for _ in range(n_runs):
        fn(*args)
    return (time.time() - t0) / n_runs

T, d = 128, 64
Q = rng.standard_normal((T, d))
K = rng.standard_normal((T, d))
V = rng.standard_normal((T, d))
t_ref = bench(attn_ref, Q, K, V, n_runs=3)
t_tile = bench(attn_tiled, Q, K, V, 32, n_runs=3)
print('ref_sec', round(t_ref, 5), 'tiled_sec', round(t_tile, 5))